In [1]:
# ── Cell 1 : Setup ───────────────────────────────────────
# Si pas encore installé :
# pip install transformers torch

from transformers import pipeline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports OK")

c:\Users\abir\OneDrive\Desktop\data-driven-audit\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports OK


In [2]:
# ── Cell 2 : Load zero-shot classifier ───────────────────
# Premier chargement : ~1.6GB téléchargés automatiquement
# Les fois suivantes : instantané (mis en cache)

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=-1,  # CPU — 1 si on a un GPU
)

# Labels en langage naturel 

LABELS = [
    "masculine-coded language promoting competition and dominance",
    "feminine-coded language promoting collaboration and care",
    "neutral professional language",
]

print("✅ Model loaded")
print(f"Labels utilisés :")
for l in LABELS: print(f"  → {l}")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


✅ Model loaded
Labels utilisés :
  → masculine-coded language promoting competition and dominance
  → feminine-coded language promoting collaboration and care
  → neutral professional language


In [3]:
# ── Cell 2 : Load zero-shot classifier ───────────────────
# Premier chargement : ~1.6GB téléchargés automatiquement
# Les fois suivantes : instantané (mis en cache)

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=-1,  # CPU — change à 0 si tu as un GPU
)

# Labels en langage naturel — c'est ici que la magie opère
LABELS = [
    "masculine-coded language promoting competition and dominance",
    "feminine-coded language promoting collaboration and care",
    "neutral professional language",
]

print("✅ Model loaded")
print(f"Labels utilisés :")
for l in LABELS: print(f"  → {l}")

✅ Model loaded
Labels utilisés :
  → masculine-coded language promoting competition and dominance
  → feminine-coded language promoting collaboration and care
  → neutral professional language


In [4]:
# ── Cell 3 : Sanity check sur exemples connus ────────────
test_cases = [
    {
        "text": "We need a competitive rockstar who dominates the market "
                "and aggressively drives results. Independent self-starter.",
        "expected": "masculine"
    },
    {
        "text": "Join our collaborative team where we nurture each other's growth. "
                "We value empathy, trust, and building meaningful relationships.",
        "expected": "feminine"
    },
    {
        "text": "You will manage quarterly reports, coordinate with stakeholders, "
                "and ensure compliance with industry standards.",
        "expected": "neutral"
    },
    {
        "text": "We challenge discrimination and actively support diverse candidates. "
                "Strong commitment to inclusion.",
        "expected": "neutral/feminine"  # le mot 'strong' mais contexte inclusif
    },
]

print("🧪 Sanity check — zero-shot vs keyword matching\n")
print(f"{'Text (truncated)':<55} {'Expected':<12} {'BERT prediction':<30} {'Score'}")
print("─" * 110)

for case in test_cases:
    result = classifier(case["text"][:300], LABELS, multi_label=False)
    top_label = result["labels"][0]
    top_score = result["scores"][0]

    # Simplify label for display
    label_short = (
        "masculine" if "masculine" in top_label else
        "feminine"  if "feminine"  in top_label else
        "neutral"
    )

    print(
        f"{case['text'][:52]+'...':<55} "
        f"{case['expected']:<12} "
        f"{label_short:<30} "
        f"{top_score:.2%}"
    )

🧪 Sanity check — zero-shot vs keyword matching

Text (truncated)                                        Expected     BERT prediction                Score
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
We need a competitive rockstar who dominates the mar... masculine    masculine                      96.71%
Join our collaborative team where we nurture each ot... feminine     feminine                       72.88%
You will manage quarterly reports, coordinate with s... neutral      feminine                       57.63%
We challenge discrimination and actively support div... neutral/feminine neutral                        59.45%


In [ ]:
# ── Cell 4 : Score sur échantillon représentatif ─────────
import time

df = pd.read_csv("../data/processed/jobs_scored.csv")

sample = (
    df.groupby("sector", group_keys=False)
    .apply(lambda x: x.sample(min(200, len(x)), random_state=42))
    .reset_index(drop=True)
)

print(f"Sample size : {len(sample)} offres")
print(sample["sector"].value_counts())

def classify_posting(text: str) -> tuple[str, float]:
    """Run zero-shot classification on a single posting."""
    truncated = str(text)[:1000]
    try:
        result = classifier(truncated, LABELS, multi_label=False)
        top_label = result["labels"][0]
        top_score = result["scores"][0]
        if "masculine" in top_label:
            return "masculine", top_score
        elif "feminine" in top_label:
            return "feminine", top_score
        else:
            return "neutral", top_score
    except Exception:
        return "neutral", 0.0

# ⏱ ~20-25 min sur CPU — laisse tourner
start = time.time()

results = []
for idx, row in tqdm(sample.iterrows(), total=len(sample), desc="Classifying"):
    label, score = classify_posting(row["description_clean"])
    results.append({
        "sector"       : row["sector"],
        "bert_label"   : label,
        "bert_score"   : round(score, 4),
        "keyword_label": row["bias_label"],
        "bias_score"   : row["bias_score"],
    })

elapsed = time.time() - start
print(f"\n⏱ Temps total : {elapsed/60:.1f} min")
print(f"✅ {len(results)} offres classifiées")

df_bert = pd.DataFrame(results)
df_bert.to_csv("../data/processed/bert_results.csv", index=False)
print("💾 bert_results.csv sauvegardé")